# Straight-line robot drift: simulation and physical validation

## Introduction
This experiment asks how far a differential-drive robot naturally drifts under equal, constant motor PWM, and which level of simulation complexity best explains its behaviour in the real world. We first establish a **sterile, deterministic baseline**, then compare it with a separate model containing motor dynamics, wheel-speed disturbances. Finally, repeated physical observations will be used to select and validate the best-fitting candidate model.

The response variable is the **signed perpendicular distance from the robot's centrepoint to the line centre**, measured after 3, 4, 5 seconds and at later one-second intervals. The centrepoint is the midpoint of the wheel axle and the body origin used by the DigitalTwin, **not the front sensor array**. Forward distance is recorded separately: a robot that stops moving can have low drift without travelling straight.

This notebook runs the actual experiments in a live Processing window through Python, overwrites selected simulation sheets in an Excel workbook, reads them with **pandas**, and plots them with **matplotlib**. The open-loop experiment and noisy extension live in `SimulationBatch.java`; the original DigitalTwin line-following experiment is separate. Its parameters are illustrative candidates, not measurements of this robot. **Real-world fitting remains pending until measured data are supplied.**

## 1. Method and measurement

The trial uses a 10 ms physics step and a 30 s duration. Both motors receive constant PWM **45 / 45**, with no motor calibration, line-following, speed matching or heading feedback. Each trial starts at rest at the origin, facing +x. The drawn line is only a visual reference for the initial heading. Time zero is immediately before the first driving interval. Each trial contains 31 samples, from 0 through 30 s. Rendering and wall-clock timing do not determine the sample times.

For a directed line passing through $(x_0,y_0)$ at heading $\phi$, centrepoint drift is

$$d(t)=-\sin\phi\,[x_c(t)-x_0]+\cos\phi\,[y_c(t)-y_0].$$

Positive values are **left** and negative values **right**, looking forward along the line. In the simulated frame, $(x_0,y_0)=(0,0)$ and $\phi=0$, so $d(t)=y_c(t)$. Distance is in millimetres and time in seconds. The experiment reads the continuous world pose, rather than the encoder estimate. Real-world data should come from an independent centrepoint measurement, such as calibrated overhead video or a measured centre marker; wheel odometry alone is not independent ground truth.

### Models and controlled conditions

| Property | Sterile DigitalTwin | Separate noisy extension |
|---|---|---|
| Drive protocol and sampling | Equal PWM 45 / 45, open loop; 0–30 s | Same protocol |
| Motor response | Instantaneous, identical linear motors | First-order lag, optional fixed left/right gain difference |
| Wheel disturbance | None | Independent Gaussian fractional target-speed noise per 10 ms tick |
| Reflectance | Not acquired or used | Not acquired or used; sensor noise has no effect |
| Starting pose | Exactly centred and aligned | Exactly centred and aligned |
| Repetition | One deterministic run | Multiple recorded seeds per candidate |

For gain difference $b$, left gain is $1-b/2$ and right gain is $1+b/2$. Motor velocity follows $v_{k+1}=v_k+(1-e^{-\Delta t/\tau})(v_{target,k}-v_k)$. Noise acts on wheel target velocity, not on the reported drift. Each motor's target is multiplied by $(1+\epsilon_k)$, with $\epsilon_k\sim N(0,\sigma_v^2)$. `sensor_noise_counts` is retained for API compatibility but is inactive in this experiment; defaults are zero. Noise magnitudes are tied to the fixed 10 ms step and must not be carried to another step unchanged.

### Key assumptions
The initial models do not include wheel slip, battery sag, floor variation, encoder faults or camera measurement error. Physical runs must apply equal, constant PWM 45 / 45 for 30 s from rest, with line following, wheel-speed balancing, motor gain compensation and heading correction disabled. Measure from the initial heading; no painted line is required. Check the hardware drive routine implements this protocol before collecting comparable trials. Preserve initial placement error in recorded data rather than silently subtracting it. Record battery/load, surface, marker-to-centre offset and timing uncertainty in the physical trial notes.

## 2. Setup

Use a Python 3.12 environment with the packages in `requirements.txt`, plus Processing 4 (Java mode). From the repository root:

```sh
python3 -m venv experiments/.venv
experiments/.venv/bin/python -m pip install -r experiments/requirements.txt
```

Select `experiments/.venv/bin/python` as this notebook's interpreter in your notebook editor. Run all cells from top to bottom. The Processing resources path below is configurable; it must contain `jdk` and `modes/java`. Python launches Java with argument lists, not shell commands. Each execution replaces the selected simulation sheets in `data/straight_line_results.xlsx`. The `real_world` sheet and unselected candidates are preserved. Close the workbook in Excel before running tests.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

# Locate the helpers even when this is the first cell run from the repository root.
EXPERIMENTS = next((folder for parent in (Path.cwd(), *Path.cwd().parents)
                    for folder in (parent, parent / "experiments")
                    if (folder / "experiment_setup.py").is_file()), None)
if EXPERIMENTS is None:
    raise FileNotFoundError("Open this notebook from the repository or experiments folder.")
if str(EXPERIMENTS) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS))
from experiment_setup import initialize_notebook
initialize_notebook(globals())

# True opens Processing windows; False runs fast without a window.
LIVE_PROCESSING = True

# Persistent Excel project; selected simulation sheets are overwritten on each run.
WORKBOOK = ROOT / "experiments/data/straight_line_results.xlsx"


### Candidate parameters

These are declared scenarios, not fitted values. `N_RUNS` controls the ensemble size for each noisy candidate; sterile always runs once. Set `N_RUNS = 1` for a quick preview. One seed per run is recorded in each sheet, and candidates use the same seed list for consistent comparisons. Select what to run using the numbered menu in the next cell group.

Never select the one random seed that happens to match the physical robot. Increase the ensemble size and expand/refine this candidate grid after collecting training observations, then freeze it before evaluating held-out runs.

The shared defaults are defined in `experiment_setup.py`. The cell below creates editable copies of the candidate dictionaries.

In [ ]:
from experiment_setup import default_candidates, COLORS, LABELS

N_RUNS = 24  # use 1 for a quick preview
BASE_SEED = 2026
FIT_TIMES = np.arange(3.0, 31.0, 1.0)
CANDIDATES = default_candidates()
# Edit candidate dictionaries here to explore other parameters.
# Example: CANDIDATES[0]["wheel_noise_fraction"] = 0.03
assert len({c["model_id"] for c in CANDIDATES}) == len(CANDIDATES)
assert all(c["model_id"] != "sterile" for c in CANDIDATES)
display(pd.DataFrame(CANDIDATES).set_index("model_id"))

### Choose what to run

Run the next cell to display a numbered terminal-style menu. **It can be the first cell you run in a fresh kernel:** it loads missing imports and default candidates automatically. Settings already defined in the notebook are preserved. You can then run section 3 and the remaining analysis cells directly. Type a number in the notebook's input prompt and press **Enter**:

1. **Run one candidate** — then choose sterile, balanced low/high noise, right-fast or left-fast from a second menu.
2. **Run all candidates sequentially** — baseline and four noisy models, one at a time.
3. **Run all candidates in parallel** — five independent Processing windows at once.

Enter **0** or **q** at either prompt to cancel. Invalid entries are retried. The menu only selects the run; section 3 starts it. Run All pauses here for your answer and then continues automatically.

Parallel trials within each model still run sequentially. Space pauses the focused window only. Closing an unfinished window, a failure, or interrupting execution stops the remaining batches; previous workbook results are preserved until the complete selection succeeds.

To change your selection, rerun this menu cell and section 3 onwards. Every successful execution of section 3 overwrites the selected simulation sheets, and plots/scores use only the selected models. For unattended execution (such as nbconvert), set `INTERACTIVE_SELECTION = False` and edit the choice in the `else` branch below.

In [ ]:
from pathlib import Path
import sys

# Locate the helpers even when this is the first cell run from the repository root.
EXPERIMENTS = next((folder for parent in (Path.cwd(), *Path.cwd().parents)
                    for folder in (parent, parent / "experiments")
                    if (folder / "experiment_setup.py").is_file()), None)
if EXPERIMENTS is None:
    raise FileNotFoundError("Open this notebook from the repository or experiments folder.")
if str(EXPERIMENTS) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS))
from experiment_setup import initialize_notebook
initialize_notebook(globals())

from run_menu import prompt_run_selection

INTERACTIVE_SELECTION = True
# Clear the previous selection so cancelling cannot reuse it accidentally.
RUN_MODE, SELECTED_CANDIDATE, ACTIVE_CANDIDATES = None, None, []
if INTERACTIVE_SELECTION:
    RUN_MODE, SELECTED_CANDIDATE = prompt_run_selection(CANDIDATES, LABELS)
else:
    RUN_MODE, SELECTED_CANDIDATE = "all_sequential", None  # unattended choice

ACTIVE_CANDIDATES = select_candidates(CANDIDATES, RUN_MODE, SELECTED_CANDIDATE)
print("Selected:", RUN_MODE, "| Models:", ", ".join(c["model_id"] for c in ACTIVE_CANDIDATES))
print(f"Trials per noisy candidate: {N_RUNS}; sterile: 1. Run section 3 to start.")

## 3. Run the simulations and update Excel
`SimulationRunner` compiles the current PDE files once. `MechanicalDriftExperiment` in `SimulationBatch.java` replaces the closed-loop step with constant equal motor demand and reuses the PDE pose recording. It never calls `Controller_c.update()` or motor calibration. The controller object only holds an encoder estimate for telemetry. `controller_mode = -1` and `drive_mode = "open_loop_equal_pwm"` identify this protocol in every Excel row.

With `LIVE_PROCESSING = True`, each batch opens a Processing sketch window showing open-loop robot motion, model/run/seed, elapsed time and centrepoint samples. The next trial starts automatically. Space pauses/resumes; T toggles tracking; +/- zoom; arrows pan. Closing an unfinished window cancels the selection and stops the notebook with an error, terminating any other running batches. The workbook is updated once all selected batches finish successfully. A failed or cancelled selection preserves previous results.

All 97 trials take roughly 49 minutes with `RUN_MODE = "all_sequential"`. With `"all_parallel"`, the five models run concurrently and normally take about 12 minutes. Set `RUN_MODE = "single"` and choose `SELECTED_CANDIDATE` to run just that model. Set `N_RUNS = 1` for a short preview (about 30 seconds in parallel or for one model), or `LIVE_PROCESSING = False` for fast headless execution. Pausing or slow rendering can extend the duration, but cannot change the fixed simulation steps or sampled results.

Incomplete trials raise an error. Equal PWM remains applied even if the robot drifts away from the reference line. The final sample is captured before stopping the motors. `centre_x_mm` and `centre_y_mm` are world centrepoint coordinates. Earlier results from line-following runs are a different protocol and must not be pooled with these trials.

In [ ]:
ACTIVE_CANDIDATES = select_candidates(CANDIDATES, RUN_MODE, SELECTED_CANDIDATE)
RESULTS = ROOT / "experiments/results/latest"
RESULTS.mkdir(parents=True, exist_ok=True)
print("Excel project:", WORKBOOK.relative_to(ROOT))
with SimulationRunner(ROOT, PROCESSING_RESOURCES, live=LIVE_PROCESSING) as simulation:
    written_models = simulation.run_workbook(
        WORKBOOK, CANDIDATES, mode=RUN_MODE, selected=SELECTED_CANDIDATE,
        runs=N_RUNS, seed=BASE_SEED,
    )
# Remove derived plots from the preceding selection; current plots are regenerated below.
for filename in ("drift_over_time.png", "centrepoint_paths.png", "physical_fit_and_residuals.png"):
    (RESULTS / filename).unlink(missing_ok=True)

# Save enough provenance to identify the precise simulation code and settings.
source_files = sorted((ROOT / "DigitalTwin").glob("*.pde")) + [
    ROOT / "experiments/SimulationBatch.java", ROOT / "experiments/simulation_runner.py",
    ROOT / "experiments/analysis.py", ROOT / "experiments/run_menu.py",
    ROOT / "experiments/experiment_setup.py", ROOT / "experiments/workbook_results.py"]
manifest = {
    "workbook": str(WORKBOOK.relative_to(ROOT)),
    "updated_sheets": written_models,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "processing_resources": str(PROCESSING_RESOURCES),
    "live_processing": LIVE_PROCESSING,
    "runs_per_noisy_candidate": N_RUNS, "base_seed": BASE_SEED,
    "run_mode": RUN_MODE,
    "selected_candidate": SELECTED_CANDIDATE if RUN_MODE == "single" else None,
    "candidates": ACTIVE_CANDIDATES, "fit_times_s": FIT_TIMES.tolist(),
    "drive_mode": "open_loop_equal_pwm", "left_pwm": 45, "right_pwm": 45,
    "motor_calibration": False, "feedback_enabled": False,
    "measurement": "robot axle/body centre; signed perpendicular world distance; +left/-right",
    "source_sha256": {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
                      for p in source_files},
    "python": sys.version,
    "packages": {name: importlib.metadata.version(name)
                 for name in ["numpy", "pandas", "matplotlib", "nbformat", "nbclient", "openpyxl"]},
}
(RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print("Excel sheets replaced and source manifest saved.")

### Read and validate with pandas
The Excel candidate sheets are the boundary between simulation and analysis. Only sheets selected in this execution are loaded, so older unselected results cannot enter comparisons. Recompute the drift from exported **centrepoint coordinates** and check it agrees with the simulator. Verify all run/time pairs, units and sampling coverage before making comparisons.

In [ ]:
results = read_simulations(WORKBOOK, written_models)
required_numeric = ["elapsed_s", "signed_drift_mm", "centre_x_mm", "centre_y_mm",
                    "forward_mm", "heading_rad", "left_pwm", "right_pwm", "controller_mode"]
assert np.isfinite(results[required_numeric].to_numpy(float)).all()
assert results.drive_mode.eq("open_loop_equal_pwm").all()
assert results.controller_mode.eq(-1).all()
assert results.left_pwm.eq(45).all() and results.right_pwm.eq(45).all()
assert not results.duplicated(["model_id", "run_id", "elapsed_s"]).any()
expected_times = np.arange(0.0, 31.0)
for _, run in results.groupby(["model_id", "run_id"]):
    np.testing.assert_array_equal(run.elapsed_s.sort_values(), expected_times)
np.testing.assert_allclose(results.signed_drift_mm, centrepoint_drift(results), atol=1e-7)
sterile = results[results.model_id == "sterile"].sort_values("elapsed_s")
if not sterile.empty:
    np.testing.assert_allclose(sterile.signed_drift_mm, 0, atol=1e-7)
    np.testing.assert_allclose(sterile.forward_mm, sterile.elapsed_s * 56.25, atol=0.1)
executed_ids = [c["model_id"] for c in ACTIVE_CANDIDATES]
assert set(results.model_id) == set(executed_ids)
for model_id in executed_ids:
    assert results[results.model_id == model_id].run_id.nunique() == (1 if model_id == "sterile" else N_RUNS)
display(results.loc[results.elapsed_s.isin([3, 4, 5]),
                    ["model_id", "run_id", "elapsed_s", "centre_x_mm", "centre_y_mm", "signed_drift_mm", "left_mm", "right_mm"]])
print(f"Validated {len(results):,} rows across {results.run_id.nunique()} complete trials.")

## 4. Results
At each time, report the ensemble mean, standard deviation and empirical 5th–95th percentile interval. The shaded intervals below show **variation across simulated runs**, not confidence intervals for the mean and not validated prediction intervals for the physical robot. With only one run for a model, standard deviation is undefined; it is left missing rather than filled with a fabricated estimate.

The Excel workbook’s **`comparison`** sheet automatically compares all stored candidates at **30 seconds**: trial count, mean signed drift, mean absolute drift, drift standard deviation, RMS drift and mean forward travel. Formulas refresh after simulation writes and recalculate in Excel. Empty candidates show “No results”; standard deviation needs two trials. Unselected sheets may contain earlier runs. This compares straightness, not agreement with physical measurements; the plots below still use only the current selection.

In [ ]:
summary = (results.groupby(["model_id", "elapsed_s"]).signed_drift_mm
           .agg(n="size", mean_mm="mean", sd_mm="std",
                p05_mm=lambda x: x.quantile(0.05), p95_mm=lambda x: x.quantile(0.95))
           .reset_index())
display(summary.loc[summary.elapsed_s.isin([3, 4, 5])].set_index(["model_id", "elapsed_s"]))

### Drift over time
Only executed models appear. All-model runs use two panels with the same scale: balanced motors and fixed wheel-gain differences. A single-model run shows that model alone. Even a symmetric noise model can produce asymmetric finite-sample means.

In [ ]:
groups = [(["sterile", "balanced_low", "balanced_high"], "Balanced motors"),
          (["sterile", "right_fast", "left_fast"], "Fixed wheel-gain difference")]
if len(executed_ids) == 1:
    groups = [(executed_ids, LABELS[executed_ids[0]])]
fig, axes = plt.subplots(1, len(groups), figsize=(6.5 * len(groups), 4.8),
                         sharey=True, squeeze=False, layout="constrained")
for ax, (model_ids, title) in zip(axes.flat, groups):
    for model_id in model_ids:
        if model_id not in executed_ids:
            continue
        part = summary[summary.model_id == model_id]
        ax.plot(part.elapsed_s, part.mean_mm, ":" if model_id == "sterile" else "-",
                color=COLORS[model_id], linewidth=2, label=LABELS[model_id])
        if part.n.min() > 1:
            ax.fill_between(part.elapsed_s, part.p05_mm, part.p95_mm,
                            color=COLORS[model_id], alpha=0.16)
    ax.set(title=title, xlabel="Travel time (s)", xlim=(0, 30), xticks=np.arange(0, 31, 5))
    ax.legend(fontsize=9, loc="best")
axes[0, 0].set_ylabel("Centrepoint drift (mm) · +left / −right")
fig.suptitle("Straight-line drift: means; 5–95% run spread shown for repeated trials")
fig.savefig(RESULTS / "drift_over_time.png", dpi=180)
plt.show()

In [ ]:
at_30 = summary[summary.elapsed_s == 30].set_index("model_id")
readout = "; ".join(f"{LABELS[model_id]}: **{at_30.loc[model_id, 'mean_mm']:+.2f} mm**"
                    for model_id in executed_ids)
display(Markdown("Mean drift at 30 s — " + readout + ". "
                 "These are simulated outcomes of the selected models, not physical measurements."))

### Centrepoint paths and progress
The next plot follows the **first declared seed**, chosen in advance, for each selected model. It is a trajectory example, not the best or worst run. Lateral scale is enlarged relative to forward scale so small deviations can be seen; this is not an equal-aspect map. The forward-distance table also checks whether low drift is caused by lack of motion.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), layout="constrained")
ax.axhline(0, color=COLORS["sterile"], linestyle=":", linewidth=2, label="Initial heading reference")
for model_id, style in zip(executed_ids, [":", "-", "--", "-.", "-"]):
    path = results[(results.model_id == model_id) & (results.seed == BASE_SEED)]
    ax.plot(path.centre_x_mm, path.centre_y_mm, style, color=COLORS[model_id], label=LABELS[model_id])
ax.set(title=f"Robot-centre paths · seed {BASE_SEED} · lateral scale enlarged",
       xlabel="Forward centre position (mm)", ylabel="Lateral centre position (mm)")
ax.legend(fontsize=9, ncols=2)
fig.savefig(RESULTS / "centrepoint_paths.png", dpi=180)
plt.show()

final = results[results.elapsed_s == 30]
progress = final.groupby("model_id").agg(
    runs=("run_id", "size"), mean_forward_mm=("forward_mm", "mean"),
    min_forward_mm=("forward_mm", "min"), max_forward_mm=("forward_mm", "max"))
progress["equal_pwm_all_samples"] = results.assign(
    equal_pwm=results.left_pwm.eq(45) & results.right_pwm.eq(45)
).groupby("model_id").equal_pwm.all()
display(progress)

## 5. Add real-world observations
Open `data/straight_line_results.xlsx` and fill the **`real_world`** sheet with measured positions, starting on row 2. Keep row 1 headers unchanged. Simulation runs never overwrite this sheet. No physical observations are fabricated here.

| Column | Meaning |
|---|---|
| `run_id` | Unique physical trial ID; repeated for that trial's timestamps |
| `split` | `train` or `test`; assign entire runs before fitting |
| `elapsed_s` | Seconds since driving begins, equal PWM applied from rest |
| `centre_x_mm`, `centre_y_mm` | Independently measured robot-centre coordinates in mm |

Use the same sample times for every run. The default fit uses 3 through 30 s; edit `FIT_TIMES` if, for example, only 3, 4 and 5 s were measured. Missing requested times, duplicate observations, nonfinite values, or mixing training and test timestamps from the same run will raise errors. The code does not extrapolate, fill missing observations with zero, or align timestamps by optimising the fit. Register the camera to the initial heading reference and establish the start event independently.

The physical line's origin and heading below can differ from the simulation's coordinate system. The signed-distance projection makes the drift comparable. For a marker away from the axle centre, first transform the tracked marker position to the centre using the measured rigid offset and robot heading. Record measurement uncertainty and retain the original tracking data.

In [ ]:
REAL_LINE_ORIGIN_MM = (0.0, 0.0)
REAL_LINE_HEADING_RAD = 0.0
real = load_observations(WORKBOOK, FIT_TIMES, REAL_LINE_ORIGIN_MM, REAL_LINE_HEADING_RAD)
if not real.empty:
    manifest["real_observations_sha256"] = hashlib.sha256(
        real.to_json(orient="split").encode()).hexdigest()
    manifest["real_line_origin_mm"] = REAL_LINE_ORIGIN_MM
    manifest["real_line_heading_rad"] = REAL_LINE_HEADING_RAD
    (RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
    display(real.head(12))
else:
    display(Markdown("**Physical fitting pending:** enter measured centrepoint positions in "
                     "the workbook's `real_world` sheet. Simulation comparisons are available now."))

## 6. Select a best-fitting model, then test it on unseen physical runs
“Best fit” here means the **lowest average empirical continuous ranked probability score (CRPS)** among the models executed in this run. Single-model mode evaluates that model only; it cannot establish a best candidate relative to models that were not run. For predictions $X_1,\ldots,X_M$ and physical observation $y$ at a given time,

$$\operatorname{CRPS}=\frac{1}{M}\sum_i|X_i-y|-\frac{1}{2M^2}\sum_i\sum_j|X_i-X_j|.$$

This score is in millimetres and rewards matching both position and spread. It reduces to absolute error for the deterministic baseline. Ranking solely by the mean trajectory could miss a bad noise amplitude. Mean-prediction RMSE and MAE are also reported, but the selection uses training CRPS only, with equal weight per physical run and requested time.

This is a finite candidate search, not proof of a globally optimal or physically identifiable parameter set. CRPS is evaluated at each time; it is not a joint likelihood for the correlated trajectory. We do not optimise an individual seed, average the sterile and noisy trajectories into a new physical model, or infer friction/wheel parameters from drift alone. Add independently measured speed/geometry constraints for stronger system identification.

Keep whole physical runs held out. After selecting a candidate on `train`, the code scores only that locked choice on `test`. If no held-out runs are available, it reports the calibration result without claiming validation.

In [ ]:
best_model = None
training_scores = pd.DataFrame()
if not real.empty:
    train = real[real.split == "train"]
    test = real[real.split == "test"]
    if train.empty:
        display(Markdown("**Fit pending:** at least one training run is required; test data are not used to choose a model."))
    else:
        training_scores = score_models(results, train, FIT_TIMES)
        display(training_scores)
        best_model = training_scores.iloc[0].model_id
        print("Selected on training CRPS:" if len(executed_ids) > 1 else
              "Evaluated model (no candidate comparison):", best_model)
        if best_model == "sterile":
            print("Selected parameters: no added noise, no motor lag, no gain imbalance.")
        else:
            display(pd.DataFrame([c for c in CANDIDATES if c["model_id"] == best_model]))
        if not test.empty:
            held_out = score_models(results[results.model_id == best_model], test, FIT_TIMES)
            print("Held-out evaluation of the locked training choice:")
            display(held_out)
        else:
            display(Markdown("**No held-out runs:** this is a training fit, not physical validation."))
else:
    print("No best-fit claim: real-world observations have not been supplied.")

### Physical comparison and residuals
When measurements are supplied, plot the selected model's ensemble and the held-out physical runs. If only training runs exist, label the plot accordingly. Residuals are measured drift minus predicted mean drift, so systematic errors remain visible.

In [ ]:
if best_model is not None:
    evaluation = real[real.split == "test"]
    evaluation_label = "Held-out physical runs"
    if evaluation.empty:
        evaluation = real[real.split == "train"]
        evaluation_label = "Training physical runs (not held out)"
    selected = summary[summary.model_id == best_model]
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True, layout="constrained")
    axes[0].plot(selected.elapsed_s, selected.mean_mm, color=COLORS[best_model],
                 label=f"{LABELS[best_model]} mean")
    axes[0].fill_between(selected.elapsed_s, selected.p05_mm, selected.p95_mm,
                         color=COLORS[best_model], alpha=0.18, label="5–95% simulated run spread")
    residual_rows = []
    for run_id, observed in evaluation.groupby("run_id"):
        aligned = observed.merge(selected[["elapsed_s", "mean_mm"]], on="elapsed_s",
                                 how="left", validate="many_to_one")
        assert aligned.mean_mm.notna().all()
        aligned["residual_mm"] = aligned.signed_drift_mm - aligned.mean_mm
        residual_rows.append(aligned)
        axes[0].plot(aligned.elapsed_s, aligned.signed_drift_mm, "o-", alpha=0.6, label=str(run_id))
        axes[1].plot(aligned.elapsed_s, aligned.residual_mm, "o-", alpha=0.6)
    axes[0].set(title=evaluation_label, ylabel="Centrepoint drift (mm)")
    axes[0].legend(fontsize=9, ncols=2)
    axes[1].axhline(0, color="#30343b", linestyle=":")
    axes[1].set(xlabel="Travel time (s)", ylabel="Observed − model mean (mm)", title="Residuals")
    fig.savefig(RESULTS / "physical_fit_and_residuals.png", dpi=180)
    plt.show()
else:
    print("Physical comparison plots will appear after measured training data are loaded.")

## 7. Interpretation and next steps
The sterile model is a control condition. Zero drift is expected from its symmetric ideal motors and aligned start, and does not establish that the real robot travels straight. Nonzero drift in the extended model demonstrates the consequences of its declared assumptions; it does not establish that those are the mechanisms affecting the real robot.

Use the simulated mean and spread together, and inspect progress and constant equal PWM before interpreting a model as successful. The one-second output can miss faster oscillations; retain higher-frequency measurements if those matter. Real camera noise should be measured separately before interpreting observed variability as mechanical variability.

Collect multiple physical trials with an independently measured centrepoint and start event, reserve whole trials for validation, then rerun this notebook. A best-fitting candidate is a provisional explanation within the tested grid. Report its held-out error and remaining systematic residuals rather than claiming the simulations and hardware are equal.

In [ ]:
if not sterile.empty:
    baseline_max = sterile.signed_drift_mm.abs().max()
    print(f"Executed sterile result: maximum absolute centrepoint drift = {baseline_max:.6f} mm.")
print("Executed models:", ", ".join(executed_ids), "| Mode:", RUN_MODE)
print(f"Saved {len(results):,} raw simulation rows across {results.run_id.nunique()} trials.")
print("Physical fit:", "pending measured data" if best_model is None else best_model)
print("Figures and provenance:", RESULTS.relative_to(ROOT))
print("Results workbook:", WORKBOOK.relative_to(ROOT))